# 99 - Group Model Comparison
Collects the final metrics saved by each model notebook (`results/final_metrics/*.csv`) and compares the optimum result of every model on the **same shared test set**.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

RESULTS_DIR = Path("../results")
METRICS_DIR = RESULTS_DIR / "final_metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

files = sorted(METRICS_DIR.glob("*.csv"))
print(len(files), "model result files found:")
for f in files:
    print("  -", f.name)
assert files, "No results yet - run the model notebooks first."

In [ ]:
results = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
members = pd.read_csv(RESULTS_DIR / 'group_members.csv')[['member', 'student_id']]
results = results.merge(members, on='student_id', how='left')
results = results.sort_values('f1', ascending=False).reset_index(drop=True)
results.index = results.index + 1
cols = ['model','member','student_id','accuracy','precision','recall','f1','roc_auc',
        'cv_f1_mean','cv_f1_std','train_time_s']
display(results[cols])
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(RESULTS_DIR / 'group_model_comparison.csv', index_label='rank')

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
long = results.melt(id_vars='model', value_vars=metrics, var_name='metric', value_name='score')
plt.figure(figsize=(11, 5))
sns.barplot(data=long, x='model', y='score', hue='metric')
plt.ylim(0.5, 1.0)
plt.title('Optimum result of each model on the shared test set')
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'group_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best = results.iloc[0]
print(f"Best model: {best['model']} ({best['member']}, {best['student_id']})")
print(f"F1 = {best['f1']:.4f} | Accuracy = {best['accuracy']:.4f} | ROC-AUC = {best['roc_auc']:.4f}")